# Visualisation Des Donnees AQUA-ATMOS

Ce notebook sert a explorer les donnees climatiques ouvertes et les donnees synthetiques qui serviront a l'entrainement.


## 1. Chargement

**Objectif**
Charger les donnees synthetiques annuelles et les historiques climatiques normalises.

**Resultat attendu**
Obtenir deux DataFrames propres, avec les types numeriques corrects et une liste claire des villes disponibles.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="talk")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SYNTHETIC_PATH = PROJECT_ROOT / "data" / "synthetic" / "synthetic_year.csv"
CLIMATE_DIR = PROJECT_ROOT / "data" / "climate" / "normalized"


def load_climate_history(climate_dir: Path) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(climate_dir.glob("*_hourly.csv")):
        frame = pd.read_csv(csv_path)
        frame["timestamp_utc"] = pd.to_datetime(frame["timestamp_utc"])
        numeric_columns = [
            "year",
            "month",
            "day",
            "hour",
            "temp_air_c",
            "hr_pct",
            "dew_point_c",
            "solar_wm2",
            "cloud_cover_pct",
        ]
        frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


synthetic_df = pd.read_csv(SYNTHETIC_PATH)
climate_df = load_climate_history(CLIMATE_DIR)

synthetic_df.head(), climate_df.head(), sorted(climate_df["city_id"].unique().tolist())


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/synthetic/synthetic_year.csv'

## 2. Vue D'Ensemble

**Objectif**
Verifier la taille, les colonnes et les statistiques descriptives principales.

**Resultat attendu**
Confirmer que les deux jeux de donnees sont exploitables avant de passer aux graphiques.


In [ ]:
print("Synthetic shape:", synthetic_df.shape)
print("Climate shape:", climate_df.shape)

display(synthetic_df.describe(include="all").T)
display(climate_df.describe(include="all").T)


## 3. Couverture Par Ville

**Objectif**
Mesurer la couverture temporelle et le volume de donnees pour chaque ville.

**Resultat attendu**
Verifier qu'aucune ville n'est sous-representee avant l'entrainement.


In [ ]:
coverage = (
    climate_df.groupby("city_id")
    .agg(
        rows=("timestamp_utc", "size"),
        start=("timestamp_utc", "min"),
        end=("timestamp_utc", "max"),
        min_temp=("temp_air_c", "min"),
        max_temp=("temp_air_c", "max"),
        min_hr=("hr_pct", "min"),
        max_hr=("hr_pct", "max"),
    )
    .sort_index()
)
display(coverage)


## 4. Distributions Climatiques

**Objectif**
Comparer les distributions de temperature, humidite relative et rayonnement solaire entre les villes.

**Resultat attendu**
Voir si le panel retenu couvre bien des regimes climatiques differents.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

sns.boxplot(data=climate_df, x="city_id", y="temp_air_c", ax=axes[0])
axes[0].set_title("Temperature par ville")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=climate_df, x="city_id", y="hr_pct", ax=axes[1])
axes[1].set_title("Humidite relative par ville")
axes[1].tick_params(axis="x", rotation=45)

sns.boxplot(data=climate_df, x="city_id", y="solar_wm2", ax=axes[2])
axes[2].set_title("Rayonnement solaire par ville")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


## 5. Profils Horaires Moyens

**Objectif**
Visualiser le cycle journalier moyen de la temperature, de l'humidite et du solaire pour chaque ville.

**Resultat attendu**
Verifier que les donnees reelles restituent des profils jour/nuit coherents.


In [ ]:
hourly_profile = (
    climate_df.groupby(["city_id", "hour"])
    .agg(
        temp_air_c=("temp_air_c", "mean"),
        hr_pct=("hr_pct", "mean"),
        solar_wm2=("solar_wm2", "mean"),
    )
    .reset_index()
)

fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)
metrics = ["temp_air_c", "hr_pct", "solar_wm2"]
titles = ["Temperature", "Humidite relative", "Rayonnement solaire"]

for ax, metric, title in zip(axes, metrics, titles):
    sns.lineplot(data=hourly_profile, x="hour", y=metric, hue="city_id", marker="o", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Heure")
    ax.legend(title="Ville", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


## 6. Saisonnalite

**Objectif**
Etudier les variations mensuelles par ville.

**Resultat attendu**
Identifier les mois favorables et defavorables a la production potentielle.


In [ ]:
monthly_profile = (
    climate_df.groupby(["city_id", "month"])
    .agg(
        temp_air_c=("temp_air_c", "mean"),
        hr_pct=("hr_pct", "mean"),
        solar_wm2=("solar_wm2", "mean"),
    )
    .reset_index()
)

fig, axes = plt.subplots(3, 1, figsize=(18, 18), sharex=True)
for ax, metric, title in zip(axes, ["temp_air_c", "hr_pct", "solar_wm2"], ["Temperature", "Humidite", "Solaire"]):
    sns.lineplot(data=monthly_profile, x="month", y=metric, hue="city_id", marker="o", ax=ax)
    ax.set_title(f"Saisonnalite - {title}")
    ax.legend(title="Ville", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


## 7. Cas Rares Et Extemes

**Objectif**
Isoler les queues de distribution qui devront etre preservees lors de l'entrainement et de l'evaluation.

**Resultat attendu**
Obtenir un tableau des seuils extremes par ville pour guider les scenarios de test.


In [ ]:
extremes = (
    climate_df.groupby("city_id")
    .agg(
        temp_p01=("temp_air_c", lambda s: s.quantile(0.01)),
        temp_p99=("temp_air_c", lambda s: s.quantile(0.99)),
        hr_p01=("hr_pct", lambda s: s.quantile(0.01)),
        hr_p99=("hr_pct", lambda s: s.quantile(0.99)),
        solar_p01=("solar_wm2", lambda s: s.quantile(0.01)),
        solar_p99=("solar_wm2", lambda s: s.quantile(0.99)),
    )
    .sort_index()
)
display(extremes)


## 8. Cibles Synthetiques

**Objectif**
Observer l'equilibre des labels produits par les regles metier.

**Resultat attendu**
Identifier les eventuels desequilibres de classes avant l'entrainement.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 6))

synthetic_df["profile_name"].value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Profils synthetiques")
synthetic_df["vcrc_state"].value_counts().sort_index().plot(kind="bar", ax=axes[1], title="Distribution VCRC")
synthetic_df["sorbent_mode"].value_counts().sort_index().plot(kind="bar", ax=axes[2], title="Modes sorbant")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## 9. Lecture Pour La Suite

**Objectif**
Resumer ce qui doit etre retenu avant de lancer le pretraitement et l'entrainement.

**Resultat attendu**
Une courte checklist pour la prochaine etape.


In [ ]:
checklist = [
    "Verifier qu'aucune ville n'est massivement sous-representee.",
    "Verifier qu'il n'y a pas de fuite temporelle entre train, validation et test.",
    "Conserver les cas extremes dans les jeux d'evaluation.",
    "Comparer plusieurs modeles sur la meme partition et les memes metriques.",
]

pd.DataFrame({"checklist_avant_entrainement": checklist})
